# AMP-CVAE Kaggle Notebook
This notebook clones the AMP-CVAE repository, installs dependencies, and runs training and generation of novel Antimicrobial Peptides.

**⚠️ IMPORTANT GPU NOTE:** If you are running this on Kaggle, please ensure your Accelerator is set to **GPU T4x2** instead of **GPU P100**. The latest versions of PyTorch no longer support the older architecture of the P100 GPU (sm_60), which will result in a CUDA error: no kernel image is available. The T4 GPUs are fully supported.

In [ ]:
!git clone https://github.com/Phan-Trung-Thuan/amp-cvae.git
%cd amp-cvae
!pip install -r requirements.txt

In [ ]:
import sys
sys.path.append('src')
import torch
import matplotlib.pyplot as plt
import pandas as pd
from dataset import get_dataloaders
from train import train_cvae
from generate import generate_peptides, predict_properties
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Train all baseline models using multiple GPUs in parallel
import concurrent.futures

csv_path = 'data/aps_database.csv'
seq_types = ['rnn', 'lstm', 'gru', 'transformer']
epochs = 100

histories = {}
models = {}

# Helper function to train a single model
def train_task(args):
    seq_type, gpu_id = args
    print(f"\n{'='*40}\nTraining {seq_type.upper()} model on GPU {gpu_id}\n{'='*40}")
    # We pass the specific device_id to the train function
    model, vocab, struct_enc, act_bin, history = train_cvae(
        csv_path, epochs=epochs, batch_size=32, seq_type=seq_type, lr=1e-3, device_id=gpu_id
    )
    return seq_type, model, vocab, struct_enc, act_bin, history

# Kaggle T4x2 provides 2 GPUs. We can train 2 models simultaneously.
gpu_ids = [0, 1] if torch.cuda.device_count() >= 2 else [0, 0]

# Split tasks into pairs to run concurrently on available GPUs
tasks = [
    (seq_types[0], gpu_ids[0]), (seq_types[1], gpu_ids[1]),
    (seq_types[2], gpu_ids[0]), (seq_types[3], gpu_ids[1])
]

# We will need these globally later for inference
vocab_global, struct_enc_global, act_bin_global = None, None, None

with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    # Submit tasks in pairs
    for i in range(0, len(tasks), 2):
        pair = tasks[i:i+2]
        futures = {executor.submit(train_task, task): task for task in pair}
        
        for future in concurrent.futures.as_completed(futures):
            seq_type, model, vocab, struct_enc, act_bin, history = future.result()
            histories[seq_type] = history
            models[seq_type] = model
            
            if vocab_global is None:
                vocab_global = vocab
                struct_enc_global = struct_enc
                act_bin_global = act_bin

# Re-assign back to the variable names expected by the rest of the notebook
vocab = vocab_global
struct_enc = struct_enc_global
act_bin = act_bin_global


In [ ]:
def plot_metrics(histories, epochs):
    metrics = ['total_loss', 'recon_loss', 'kl_loss', 'charge_loss', 'hydro_loss', 'struct_loss', 'activity_loss']
    fig, axes = plt.subplots(len(metrics), 2, figsize=(15, 5*len(metrics)))
    
    for i, metric in enumerate(metrics):
        # Train
        ax_train = axes[i, 0]
        for seq_type, hist in histories.items():
            ax_train.plot(range(1, epochs+1), hist[f'train_{metric}'], label=seq_type)
        ax_train.set_title(f'Train {metric}')
        ax_train.set_xlabel('Epochs')
        ax_train.set_ylabel(metric)
        ax_train.legend()
        ax_train.grid(True)
        
        # Val
        ax_val = axes[i, 1]
        for seq_type, hist in histories.items():
            ax_val.plot(range(1, epochs+1), hist[f'val_{metric}'], label=seq_type)
        ax_val.set_title(f'Validation {metric}')
        ax_val.set_xlabel('Epochs')
        ax_val.set_ylabel(metric)
        ax_val.legend()
        ax_val.grid(True)
        
    plt.tight_layout()
    plt.show()

plot_metrics(histories, epochs)

In [ ]:
# Create results table
results = []
for seq_type, hist in histories.items():
    res = {'Model': seq_type.upper()}
    for k in hist.keys():
        res[k] = hist[k][-1] # Take the last epoch value
    results.append(res)

df_results = pd.DataFrame(results)
print("Training Results Table (Final Epoch):\n")
print(df_results.to_markdown(index=False))


In [ ]:
# Generate new peptides and verify properties for ALL models
target_charge = 3.0
target_hydro = 45.0
target_struct = 'alpha-helix'
target_activity = ['Anti-Gram+ & Gram-']

for seq_type, model in models.items():
    print(f"\n{'='*40}\nInference with {seq_type.upper()} model\n{'='*40}")
    
    generated_seqs = generate_peptides(
        model, 
        vocab, 
        target_charge=target_charge, 
        target_hydro=target_hydro, 
        target_struct=target_struct, 
        target_activity=target_activity, 
        struct_enc=struct_enc,
        act_bin=act_bin,
        num_samples=3, 
        max_len=50
    )
    
    print("Generated Peptides:")
    for i, seq in enumerate(generated_seqs):
        print(f"{i+1}: {seq}")
        
    print("\n--- Verifying Properties with Encoder ---")
    predictions = predict_properties(model, vocab, generated_seqs, struct_enc, act_bin)
    for p in predictions:
        print(f"\nSequence: {p['sequence']}")
        print(f"  Predicted Charge: {p['charge']:.2f} (Target: {target_charge})")
        print(f"  Predicted Hydrophobicity: {p['hydrophobicity']:.2f} (Target: {target_hydro})")
        print(f"  Predicted Structure: {p['structure']} (Target: {target_struct})")
        print(f"  Predicted Activities: {p['activities']} (Target: {target_activity})")
